# Train Helmet Detector — Smart Traffic Violation Detection

Fine-tunes YOLOv8n on an Indian helmet-detection dataset and produces `best.pt` to drop into the deployed system.

**Why fine-tune instead of train from scratch?** YOLOv8n already knows what cars/people/motorcycles look like (COCO pretraining). We're teaching it two new classes: *with helmet* / *without helmet*. Fine-tuning takes ~30 min on a Colab T4 GPU; training from scratch would take days.

**Run on:** Google Colab → Runtime → Change runtime type → T4 GPU.

---

## Step 1 — Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

## Step 2 — Pull the dataset from Roboflow

We use [helmet-license-plate-detection-gevlq](https://universe.roboflow.com/cdio-zmfmj/helmet-lincense-plate-detection-gevlq) — ~2000 Indian-context images with helmet/no-helmet/plate/rider classes.

1. Sign up at https://app.roboflow.com (free).
2. Settings → API → copy your private API key.
3. Paste below. **Don't commit your key** — Colab is fine, just don't push the notebook with the key filled in.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass('Roboflow API key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('cdio-zmfmj').project('helmet-lincense-plate-detection-gevlq')
dataset = project.version(1).download('yolov8')
print('Dataset at:', dataset.location)

## Step 3 — Train

Hyperparameters worth knowing:

- `epochs=50` — usually enough for 2–4 class fine-tunes on ~2k images. Bump to 100 if mAP plateaus low.
- `imgsz=640` — YOLOv8's default; matches what our backend uses at inference time.
- `batch=16` — fits in T4's 15GB. Drop to 8 if you OOM.
- `patience=15` — early stop if val mAP doesn't improve.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # pretrained COCO weights
results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    name='helmet_v1',
)

## Step 4 — Evaluate

Look at mAP@0.5 on the validation set. >0.7 is solid for this problem; >0.85 is excellent.

In [ ]:
metrics = model.val()
print('mAP@0.5      :', round(metrics.box.map50, 3))
print('mAP@0.5:0.95 :', round(metrics.box.map, 3))
print('Per-class mAP@0.5:', metrics.box.maps)

## Step 5 — Sanity-check predictions

In [ ]:
from PIL import Image
import glob, random

test_imgs = glob.glob(f'{dataset.location}/test/images/*.jpg')
random.shuffle(test_imgs)
for img in test_imgs[:3]:
    pred = model(img)[0]
    Image.fromarray(pred.plot()).save(f'/tmp/pred_{img.split("/")[-1]}')
print('Predictions saved to /tmp/pred_*.jpg — view in the Colab files panel.')

## Step 6 — Export `best.pt` and deploy

The trained weights are at `runs/detect/helmet_v1/weights/best.pt`. Download it and drop into the deployed system:

1. **Download** below.
2. **Local test:** copy to `ml/models/helmet_detector.pt` in the repo, then `pytest backend/tests/test_detector.py`.
3. **Deploy to Railway:** upload to the backend service's volume at `/data/models/helmet_detector.pt`, then redeploy the worker so it picks up the new file.

In [ ]:
from google.colab import files
files.download('runs/detect/helmet_v1/weights/best.pt')